# Topic 1: Structured Streaming — Basics

> **Phase 6 → Week 10 → Topic 1**
>
> Prerequisites: Week 8-9 (DataFrames, ETL patterns)

---

## What is Spark Structured Streaming?

Structured Streaming treats a **live data stream as an unbounded table** that keeps growing. You write the same DataFrame API you already know — Spark handles the streaming complexity.

```
Batch (Week 8-9):                      Structured Streaming:
  ┌──────────────────┐                   ┌──────────────────────────────────┐
  │  Static Table    │                   │  Unbounded Input Table           │
  │  (fixed rows)    │                   │  (new rows keep arriving)        │
  │  row 1           │                   │  row 1  row 2  row 3  row 4 ...  │
  │  row 2           │                   │  t=0    t=1    t=2    t=3        │
  │  row 3           │                   └──────────────────────────────────┘
  └──────────────────┘                            ↓ query ↓
         ↓ query ↓                      ┌──────────────────────────────────┐
  ┌──────────────────┐                  │  Result Table (continuously      │
  │  Result Table    │                  │  updated as new data arrives)    │
  │  (snapshot)      │                  └──────────────────────────────────┘
  └──────────────────┘                            ↓ output ↓
                                        ┌─────────────────────────────┐
                                        │  Sink (Kafka / file / DB)   │
                                        └─────────────────────────────┘
```

---

## Micro-batch vs Continuous Processing

```
Micro-batch (default):                  Continuous Processing (experimental):
  Spark batches records collected          True record-by-record processing.
  over a time interval (trigger),          Latency: ~1 ms.
  processes them as a mini-batch.          Only supports simple map/filter.
  Latency: ~100ms–few seconds.             Not used in production yet.
  Exactly-once: YES.
  Full DataFrame API: YES.
  → This is what you use in production.
```

---

## Trigger Options

```python
from pyspark.sql.streaming import Trigger

# 1. Default (micro-batch ASAP) — process next batch immediately after previous finishes
.trigger(processingTime='0 seconds')   # or omit trigger entirely

# 2. Fixed interval — process a batch every N seconds (skip if no data)
.trigger(processingTime='30 seconds')
.trigger(processingTime='5 minutes')

# 3. Once — process all available data now, then stop (like a batch job)
.trigger(once=True)

# 4. AvailableNow — like Once but processes data in multiple batches (Spark 3.3+)
.trigger(availableNow=True)
```

---

## Output Modes

```
Mode         When to use                        What is written each trigger
──────────── ────────────────────────────────── ────────────────────────────────
append       No aggregations / watermarked agg  Only NEW rows added since last trigger
complete     Aggregations WITHOUT watermark      Entire result table every trigger
update       Aggregations WITH watermark         Only CHANGED rows since last trigger
```

---

## Interview Cheat Sheet

**Q: What is Structured Streaming and how is it different from DStream?**
> Structured Streaming is Spark's high-level streaming API built on top of the DataFrame/Dataset API and Catalyst Optimizer. It models a stream as an unbounded table. DStreams (Spark Streaming) were RDD-based micro-batches with a separate API. Structured Streaming is strictly better: same API as batch, fault tolerance via checkpointing, exactly-once semantics, watermarking for late data, and access to all DataFrame optimizations.

**Q: What is a trigger in Structured Streaming?**
> A trigger defines how often Spark checks for new data and processes a micro-batch. `processingTime='30 seconds'` fires every 30 seconds. If processing finishes early, Spark waits for the next interval. If it takes longer, the next batch starts immediately after. `once=True` processes all available data then stops — useful for scheduled "streaming" jobs.

**Q: What are the three output modes?**
> - **Append**: write only new rows. Safe for non-aggregated queries and watermarked aggregations. Default.
> - **Complete**: rewrite the entire result table every trigger. Required for aggregations without watermark. Expensive for large state.
> - **Update**: write only rows that changed since last trigger. Most efficient for aggregations with watermark.

In [1]:
from pyspark.sql import SparkSession
from pyspark.sql import functions as F
from pyspark.sql.types import *
import time

spark = SparkSession.builder \
    .appName("Week10 - Structured Streaming Basics") \
    .master("local[4]") \
    .config("spark.sql.shuffle.partitions", "4") \
    .getOrCreate()

spark.sparkContext.setLogLevel("WARN")
print("Spark", spark.version)

Setting default log level to "WARN".
To adjust logging level use sc.setLogLevel(newLevel). For SparkR, use setLogLevel(newLevel).


26/05/16 08:47:45 WARN NativeCodeLoader: Unable to load native-hadoop library for your platform... using builtin-java classes where applicable


Spark 3.5.3


## Part 1: Your First Streaming Query

We use the built-in `rate` source — generates rows at a fixed rate with no external dependencies. Perfect for learning.

```
rate source output:
  timestamp               | value
  ----------------------- | -----
  2024-01-01 10:00:00.000 | 0
  2024-01-01 10:00:01.000 | 1
  2024-01-01 10:00:02.000 | 2
  ...one row per second...
```

In [2]:
# readStream creates a streaming DataFrame — same API as read, just streaming
rate_stream = spark.readStream \
    .format("rate") \
    .option("rowsPerSecond", 5) \
    .load()

print("Schema of the rate source:")
rate_stream.printSchema()
print("Is streaming:", rate_stream.isStreaming)

Schema of the rate source:
root
 |-- timestamp: timestamp (nullable = true)
 |-- value: long (nullable = true)

Is streaming: True


In [3]:
# Apply a transformation — same as batch
processed = rate_stream \
    .withColumn("batch", F.col("value") % 3) \
    .withColumn("label", F.when(F.col("batch") == 0, "A")
                          .when(F.col("batch") == 1, "B")
                          .otherwise("C"))

# writeStream: the trigger that starts the query
# format="memory" writes to an in-memory table we can query
# outputMode="append" only writes new rows
query = processed.writeStream \
    .format("memory") \
    .queryName("rate_demo") \
    .outputMode("append") \
    .trigger(processingTime="1 second") \
    .start()

print("Query started. Name:", query.name, "| Active:", query.isActive)
time.sleep(5)   # let it run for 5 seconds

print("\nRows collected so far:")
spark.sql("SELECT * FROM rate_demo ORDER BY timestamp").show(10)

time.sleep(5)
print("After 10 seconds total:")
spark.sql("SELECT count(*) as total_rows FROM rate_demo").show()

query.stop()
print("Query stopped.")

26/05/16 08:47:50 WARN ResolveWriteToStream: Temporary checkpoint location created which is deleted normally when the query didn't fail: /tmp/temporary-e5cd27f8-a9e7-4f32-a4c9-3add52099c63. If it's required to delete it under any circumstances, please set spark.sql.streaming.forceDeleteTempCheckpointLocation to true. Important to know deleting temp checkpoint folder is best effort.
26/05/16 08:47:50 WARN ResolveWriteToStream: spark.sql.adaptive.enabled is not supported in streaming DataFrames/Datasets and will be disabled.


Query started. Name: rate_demo | Active: True


26/05/16 08:47:53 WARN ProcessingTimeExecutor: Current batch is falling behind. The trigger interval is 1000 milliseconds, but spent 1937 milliseconds


26/05/16 08:47:55 WARN ProcessingTimeExecutor: Current batch is falling behind. The trigger interval is 1000 milliseconds, but spent 1871 milliseconds



Rows collected so far:


+--------------------+-----+-----+-----+
|           timestamp|value|batch|label|
+--------------------+-----+-----+-----+
|2026-05-16 08:47:...|    0|    0|    A|
|2026-05-16 08:47:...|    1|    1|    B|
|2026-05-16 08:47:...|    2|    2|    C|
|2026-05-16 08:47:...|    3|    0|    A|
|2026-05-16 08:47:...|    4|    1|    B|
|2026-05-16 08:47:...|    5|    2|    C|
|2026-05-16 08:47:...|    6|    0|    A|
|2026-05-16 08:47:...|    7|    1|    B|
|2026-05-16 08:47:...|    8|    2|    C|
|2026-05-16 08:47:...|    9|    0|    A|
+--------------------+-----+-----+-----+
only showing top 10 rows



After 10 seconds total:


+----------+
|total_rows|
+----------+
|        50|
+----------+

Query stopped.


## Part 2: Query Status and Progress

In [4]:
import json

stream2 = spark.readStream.format("rate").option("rowsPerSecond", 10).load()

q2 = stream2 \
    .withColumn("doubled", F.col("value") * 2) \
    .writeStream \
    .format("memory") \
    .queryName("q2_demo") \
    .outputMode("append") \
    .start()

time.sleep(3)

# Query status: what is it doing right now?
print("Status:", q2.status)

# Recent progress: how fast is it processing?
progress = q2.lastProgress
if progress:
    print("\nLast micro-batch:")
    print(f"  Batch ID:      {progress.get('batchId', 'N/A')}")
    print(f"  Input rows:    {progress.get('numInputRows', 0)}")
    print(f"  Rows/sec in:   {progress.get('inputRowsPerSecond', 0.0):.1f}")
    print(f"  Rows/sec out:  {progress.get('processedRowsPerSecond', 0.0):.1f}")
    print(f"  Trigger gap:   {progress.get('triggerExecution', {}).get('triggerDuration', 'N/A')} ms")

q2.stop()

26/05/16 08:48:03 WARN ResolveWriteToStream: Temporary checkpoint location created which is deleted normally when the query didn't fail: /tmp/temporary-be071cb3-18bf-4b38-8749-4c77c9a08c63. If it's required to delete it under any circumstances, please set spark.sql.streaming.forceDeleteTempCheckpointLocation to true. Important to know deleting temp checkpoint folder is best effort.
26/05/16 08:48:03 WARN ResolveWriteToStream: spark.sql.adaptive.enabled is not supported in streaming DataFrames/Datasets and will be disabled.


Status: {'message': 'Waiting for data to arrive', 'isDataAvailable': False, 'isTriggerActive': False}

Last micro-batch:
  Batch ID:      2
  Input rows:    10
  Rows/sec in:   909.1
  Rows/sec out:  22.0
  Trigger gap:   N/A ms


## Part 3: Output Modes — Append vs Complete vs Update

In [5]:
# COMPLETE mode: the entire result table is rewritten every trigger
# Required for aggregations without watermark
stream3 = spark.readStream.format("rate").option("rowsPerSecond", 10).load()

# Count rows per label (A/B/C)
counts = stream3 \
    .withColumn("label", F.element_at(F.array(F.lit("A"), F.lit("B"), F.lit("C")),
                                       (F.col("value") % 3 + 1).cast("int"))) \
    .groupBy("label") \
    .count()

q_complete = counts.writeStream \
    .format("memory") \
    .queryName("label_counts") \
    .outputMode("complete") \
    .trigger(processingTime="2 seconds") \
    .start()

for i in range(3):
    time.sleep(3)
    print(f"--- Snapshot at t={(i+1)*3}s ---")
    spark.sql("SELECT * FROM label_counts ORDER BY label").show()

q_complete.stop()
print("Note: complete mode rewrites the ENTIRE table every trigger — counts keep increasing")

26/05/16 08:48:07 WARN ResolveWriteToStream: Temporary checkpoint location created which is deleted normally when the query didn't fail: /tmp/temporary-dce91d14-0033-4d47-bd03-0052eff4f2b2. If it's required to delete it under any circumstances, please set spark.sql.streaming.forceDeleteTempCheckpointLocation to true. Important to know deleting temp checkpoint folder is best effort.
26/05/16 08:48:07 WARN ResolveWriteToStream: spark.sql.adaptive.enabled is not supported in streaming DataFrames/Datasets and will be disabled.


--- Snapshot at t=3s ---


+-----+-----+
|label|count|
+-----+-----+
|    A|    4|
|    B|    3|
|    C|    3|
+-----+-----+



--- Snapshot at t=6s ---


+-----+-----+
|label|count|
+-----+-----+
|    A|   14|
|    B|   13|
|    C|   13|
+-----+-----+



--- Snapshot at t=9s ---


+-----+-----+
|label|count|
+-----+-----+
|    A|   27|
|    B|   27|
|    C|   26|
+-----+-----+

Note: complete mode rewrites the ENTIRE table every trigger — counts keep increasing


## Part 4: Trigger Types — processingTime vs once vs availableNow

In [6]:
print("""
Trigger Summary:
════════════════════════════════════════════════════════════════

1. default (processingTime='0 seconds')
   → Start next batch immediately after previous finishes
   → Lowest latency, highest throughput
   → Use for: near-real-time requirements

2. processingTime='30 seconds'
   → Wait 30s between batches
   → If batch finishes in 5s, wait 25s more
   → If batch takes 40s, next starts immediately
   → Use for: cost savings (fewer batches), downstream systems that can't handle
              high-frequency writes

3. once=True
   → Process ALL available data in ONE batch, then stop
   → Streaming query with batch semantics
   → Use for: scheduled jobs (Airflow triggers this every hour)

4. availableNow=True  (Spark 3.3+)
   → Process all available data in MULTIPLE batches (controlled parallelism)
   → Better than once=True for large backlogs
   → Use for: incremental daily loads with large catch-up needs
════════════════════════════════════════════════════════════════
""")

# Demo: trigger(once=True) — process all available data then stop
import os, shutil
STREAM_IN = "/tmp/stream_input"
STREAM_OUT = "/tmp/stream_output"
for p in [STREAM_IN, STREAM_OUT]:
    if os.path.exists(p): shutil.rmtree(p)
    os.makedirs(p)

# Write some files to simulate an input directory
for i in range(3):
    batch = spark.createDataFrame(
        [(f"E{j+i*10:03d}", float(j*10)) for j in range(5)],
        ["event_id", "value"]
    )
    batch.coalesce(1).write.mode("overwrite").json(f"{STREAM_IN}/batch_{i}")

print("Input files written.")

event_schema = StructType([
    StructField("event_id", StringType()),
    StructField("value", DoubleType()),
])

if os.path.exists("/tmp/checkpoint_once"): shutil.rmtree("/tmp/checkpoint_once")

file_stream = spark.readStream \
    .schema(event_schema) \
    .json(STREAM_IN)

q_once = file_stream \
    .withColumn("doubled", F.col("value") * 2) \
    .writeStream \
    .format("parquet") \
    .option("path", STREAM_OUT) \
    .option("checkpointLocation", "/tmp/checkpoint_once") \
    .trigger(once=True) \
    .start()

q_once.awaitTermination()  # blocks until done (since once=True auto-stops)
print("trigger(once=True) completed.")
import glob
pq_files = glob.glob(STREAM_OUT + "/**/*.parquet", recursive=True)
if pq_files:
    print("Rows written:", spark.read.parquet(STREAM_OUT).count())
else:
    print("Output dir:", os.listdir(STREAM_OUT) if os.path.exists(STREAM_OUT) else "empty")


Trigger Summary:
════════════════════════════════════════════════════════════════

1. default (processingTime='0 seconds')
   → Start next batch immediately after previous finishes
   → Lowest latency, highest throughput
   → Use for: near-real-time requirements

2. processingTime='30 seconds'
   → Wait 30s between batches
   → If batch finishes in 5s, wait 25s more
   → If batch takes 40s, next starts immediately
   → Use for: cost savings (fewer batches), downstream systems that can't handle
              high-frequency writes

3. once=True
   → Process ALL available data in ONE batch, then stop
   → Streaming query with batch semantics
   → Use for: scheduled jobs (Airflow triggers this every hour)

4. availableNow=True  (Spark 3.3+)
   → Process all available data in MULTIPLE batches (controlled parallelism)
   → Better than once=True for large backlogs
   → Use for: incremental daily loads with large catch-up needs
════════════════════════════════════════════════════════════════


Input files written.


trigger(once=True) completed.
Output dir: ['_spark_metadata']


26/05/16 08:48:20 WARN ResolveWriteToStream: spark.sql.adaptive.enabled is not supported in streaming DataFrames/Datasets and will be disabled.


## Part 5: Checkpointing — Fault Tolerance

In [7]:
print("""
Checkpointing:
════════════════════════════════════════════════════════════════
Structured Streaming guarantees exactly-once processing via checkpoints.

The checkpoint directory stores:
  /checkpoint/
    offsets/    ← what data was READ (Kafka offset, file processed, etc.)
    commits/    ← what batches were COMMITTED (written to sink)
    state/      ← stateful aggregation state (groupBy counts, window state)
    metadata    ← query metadata

On restart:
  1. Spark reads the offsets directory to know where it left off
  2. Re-reads only the data it hasn't committed yet
  3. Recomputes the state from the state directory
  4. Resumes from the last committed batch

Rule: Every production streaming query MUST have a checkpointLocation.
Without it: restart = reprocess everything from the beginning.

# How to set it:
query = df.writeStream \\
    .option("checkpointLocation", "/hdfs/checkpoints/my_query") \\
    .format("delta") \\
    .start("/delta/output_table")

# Checkpoint location must be durable (HDFS, S3, ADLS — not local disk in prod)
════════════════════════════════════════════════════════════════
""")


Checkpointing:
════════════════════════════════════════════════════════════════
Structured Streaming guarantees exactly-once processing via checkpoints.

The checkpoint directory stores:
  /checkpoint/
    offsets/    ← what data was READ (Kafka offset, file processed, etc.)
    commits/    ← what batches were COMMITTED (written to sink)
    state/      ← stateful aggregation state (groupBy counts, window state)
    metadata    ← query metadata

On restart:
  1. Spark reads the offsets directory to know where it left off
  2. Re-reads only the data it hasn't committed yet
  3. Recomputes the state from the state directory
  4. Resumes from the last committed batch

Rule: Every production streaming query MUST have a checkpointLocation.
Without it: restart = reprocess everything from the beginning.

# How to set it:
query = df.writeStream \
    .option("checkpointLocation", "/hdfs/checkpoints/my_query") \
    .format("delta") \
    .start("/delta/output_table")

# Checkpoint location mu

## Exercises

1. Start a `rate` stream at 20 rows/second. Add a column `parity` = "even" if value is divisible by 2, else "odd". Write to memory sink with `append` mode. After 5 seconds, query the in-memory table and show counts of even vs odd.
2. Start a streaming aggregation that counts rows per second. Use `complete` mode. Observe how the count grows every trigger. Then switch to `update` mode — what changes?
3. Explain why you cannot use `append` mode for `groupBy().count()` without a watermark.
4. Create a file-based stream with `trigger(once=True)`. Add 3 files to the input directory. Run the query. Add 3 more files. Re-run the query with the same checkpoint location. Verify only the 3 new files are processed (not the original 3).
5. What is stored in the `offsets/` vs `commits/` subdirectory of a checkpoint?